In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Directory containing the CSV files
data_dir = "Biometrics_4HZ"

# Initialize lists to store DataFrames and user/session info
dataframes = []
user_session_df = None

# Loop through all files in the directory
for filename in os.listdir(data_dir):
    if filename.endswith(".csv"):  # Check for CSV files
        file_path = os.path.join(data_dir, filename)
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Separate user and session columns for the last occurrence
        if 'user' in df.columns and 'session' in df.columns:
            if user_session_df is None:
                # Keep user and session from the first file temporarily
                user_session_df = df[['user', 'session']]
            # Remove user and session from current dataframe
            df = df.drop(['user', 'session'], axis=1)
        
        dataframes.append(df)

# Concatenate all DataFrames and add user/session at the end
if dataframes:
    # Concatenate all dataframes without user/session
    combined_data = pd.concat(dataframes, axis=1)
    
    # Add user and session columns at the end
    if user_session_df is not None:
        combined_data = pd.concat([combined_data, user_session_df], axis=1)
    
else:
    print("No CSV files found in the directory")

## Labeling ##

In [3]:
labels = pd.read_csv("../Labels.csv", header=None)

# Rename columns for clarity
labels.columns = ["user_session"] + [f"minute_{i}" for i in range(1, 11)]


# Initialize parameters
sampling_rate = 4  # sampling rate in Hz
seconds_per_interval = 4 * 60  # 4 minutes label rate
samples_per_interval = sampling_rate * seconds_per_interval

# Create an empty dataframe for expanded labels
expanded_labels = []

# Process each user-session
for user_session, group in combined_data.groupby(["user", "session"]):
    user, session = user_session  # Extract user and session
    user_session_label = f"{user}_{session}"
    
    if user_session_label in labels["user_session"].values:
        # Get the corresponding label row
        label_row = labels[labels["user_session"] == user_session_label].iloc[0, 1:]
        
        # Expand labels
        label_list = []
        for label in label_row:
            label_list.extend([label] * samples_per_interval)
        
        # Match to the size of the data
        label_list = label_list[:len(group)]  # Truncate to match data length
        if len(label_list) < len(group):
            label_list.extend([np.nan] * (len(group) - len(label_list)))  # Fill missing labels
        
        # Append expanded labels
        expanded_labels.extend([[user, session, lbl] for lbl in label_list])
    else:
        # Handle missing labels
        expanded_labels.extend([[user, session, np.nan]] * len(group))

# Convert expanded labels to a dataframe
labels_df = pd.DataFrame(expanded_labels, columns=["user", "session", "label"])

# Concatenate labels with combined_data
labeled_data = pd.concat([combined_data.reset_index(drop=True), labels_df["label"].reset_index(drop=True)], axis=1)

# Save the labeled data
labeled_data.to_csv("Biometrics_4HZ/labeled_combined_biometric_data_4HZ.csv", index=False)

